In [4]:
# import libraries
import numpy as np # for numerical operations
import pandas as pd # for data manipulation and analysis
import tensorflow as tf # for building and training the model
from tensorflow.keras.preprocessing.text import Tokenizer # for text preprocessing
from tensorflow.keras.preprocessing.sequence import pad_sequences # for padding sequences to a fixed length
from tensorflow.keras import models,layers # for building the model
from tensorflow.keras.callbacks import EarlyStopping # for early stopping during training
from sklearn.model_selection import train_test_split # for splitting the data into training and testing sets


In [5]:
# load data 
data = {
    "text": [
        "I love this movie",
        "This film was terrible",
        "I enjoyed the plot",
        "The acting was bad",
        "What a fantastic experience",
        "I hated the ending"
    ],
    "label": [1, 0, 1, 0, 1, 0] # 1 for positive sentiment, 0 for negative sentiment
}
df=pd.DataFrame(data)
print(df)

                          text  label
0            I love this movie      1
1       This film was terrible      0
2           I enjoyed the plot      1
3           The acting was bad      0
4  What a fantastic experience      1
5           I hated the ending      0


In [6]:
# preprocess the text data
def clean_text(text):
    text=text.lower() # convert to lowercase
    return text

In [7]:
df['text']=df['text'].apply(clean_text)
print(df)

                          text  label
0            i love this movie      1
1       this film was terrible      0
2           i enjoyed the plot      1
3           the acting was bad      0
4  what a fantastic experience      1
5           i hated the ending      0


In [8]:
# train test split
X_train,X_test,y_train,y_test=train_test_split(df['text'],df['label'],test_size=0.2,random_state=42,stratify=df['label'])

In [9]:
# tokenization
VOCAB_SIZE=1000
max_length=10

tokenizer=Tokenizer(num_words=VOCAB_SIZE,oov_token='<OOV>') # oov_token is used to handle out-of-vocabulary words
tokenizer.fit_on_texts(X_train) # fit the tokenizer on the training data

X_train_seq=tokenizer.texts_to_sequences(X_train) # convert text to sequences
X_test_seq=tokenizer.texts_to_sequences(X_test)

In [10]:
# padding 
X_train_pad=pad_sequences(X_train_seq,maxlen=max_length,padding='post') # pad the sequences to a fixed length
X_test_pad=pad_sequences(X_test_seq,maxlen=max_length,padding='post')

In [12]:
X_train_pad

array([[ 2,  4,  3,  5,  0,  0,  0,  0,  0,  0],
       [ 6,  7,  8,  9,  0,  0,  0,  0,  0,  0],
       [10, 11,  3, 12,  0,  0,  0,  0,  0,  0],
       [13, 14,  2, 15,  0,  0,  0,  0,  0,  0]])

In [16]:
# model building (LSTM)    # EB(L)DDD
model=models.Sequential([
    layers.Embedding(VOCAB_SIZE, 16, input_length=max_length), # embedding layer to convert words to dense vectors  
    layers.Bidirectional(
        layers.LSTM(64, dropout=0.2, recurrent_dropout=0.2)
    ),
    layers.Dense(64, activation='relu') ,# output layer for binary classification
    layers.Dropout(0.5), # dropout layer to prevent overfitting
    layers.Dense(1, activation='sigmoid') # output layer for binary classification
])

In [17]:
# compile the model 
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

In [18]:
model.summary()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding_2 (Embedding)     (None, 10, 16)            16000     
                                                                 
 bidirectional_2 (Bidirectio  (None, 128)              41472     
 nal)                                                            
                                                                 
 dense_3 (Dense)             (None, 64)                8256      
                                                                 
 dropout_1 (Dropout)         (None, 64)                0         
                                                                 
 dense_4 (Dense)             (None, 1)                 65        
                                                                 
Total params: 65,793
Trainable params: 65,793
Non-trainable params: 0
__________________________________________________

In [20]:
# train
early_stop=EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True) # early stopping callback to prevent overfitting
model.fit(
    X_train_pad, y_train, 
    epochs=20, 
    batch_size=32, 
    validation_data=(X_test_pad, y_test), 
    callbacks=[early_stop]
)

Epoch 1/20
1/1 [==============================] - 16s 16s/step - loss: 0.6943 - accuracy: 0.2500 - val_loss: 0.6930 - val_accuracy: 0.5000
Epoch 2/20
1/1 [==============================] - 0s 113ms/step - loss: 0.6934 - accuracy: 0.5000 - val_loss: 0.6931 - val_accuracy: 0.5000
Epoch 3/20
1/1 [==============================] - 0s 120ms/step - loss: 0.6953 - accuracy: 0.5000 - val_loss: 0.6932 - val_accuracy: 0.5000
Epoch 4/20
1/1 [==============================] - 0s 148ms/step - loss: 0.6903 - accuracy: 0.5000 - val_loss: 0.6930 - val_accuracy: 0.5000


In [21]:
# evaluate
loss,acc=model.evaluate(X_test_pad, y_test)
print(f'Test Loss: {loss:.4f}, Test Accuracy: {acc:.4f}')

1/1 [==============================] - 0s 111ms/step - loss: 0.6930 - accuracy: 0.5000
Test Loss: 0.6930, Test Accuracy: 0.5000


In [22]:
# prediction
def predict_sentiment(text):
    text=clean_text(text)
    seq=tokenizer.texts_to_sequences([text]) # convert text to sequence
    pad=pad_sequences(seq,maxlen=max_length,padding='post') # pad the sequence
    pred=model.predict(pad)[0][0] # predict the sentiment

    if pred>0.5:
        return 'Positive'   
    else:
        return 'Negative'

In [24]:
# test 
prediction=predict_sentiment("I really hate this movie")
print(prediction)

1/1 [==============================] - 0s 76ms/step
Negative
